# Winnipeg Transit Performance Dataset Cleaning & Feature Engineering

This notebook cleans and standardizes raw Winnipeg Transit on-time performance data.  
The prepared dataset will later be integrated with weather data and loaded into a SQL data warehouse for analytical reporting.


## Load Raw Data

This step loads the original transit performance dataset obtained from the City of Winnipeg open data portal.


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../../data_raw/bus/winnipeg_otp.csv')
df

,Route Number,Route Name,Route Destination,Day Type,Day,Time Period,Early Stops,Late Stops,On-Time Stops,Key
0,16,Selkirk-Osborne,Tyndall Park via Manitoba,Weekday,2024 Apr 30 12:00:00 AM,09:00-16:00,123,266,408,16##Tyndall Park via Manitoba##20240430000000#...
1,28,Brookside Express,City Hall,Weekday,2024 May 02 12:00:00 AM,16:00-18:30,24,218,187,28##City Hall##20240502000000##Weekday##16:00-...
2,77,Crosstown North,Polo Park,Weekday,2024 May 03 12:00:00 AM,18:30-22:30,59,42,431,77##Polo Park##20240503000000##Weekday##18:30-...
3,694,Wildwood,Seel Station via Wildwood,Weekday,2024 May 03 12:00:00 AM,09:00-16:00,4,0,12,694##Seel Station via Wildwood##20240503000000...
4,676,Bridgwater/River Road,Centre Street via Bridgwater,Weekday,2024 May 03 12:00:00 AM,09:00-16:00,4,58,70,676##Centre Street via Bridgwater##20240503000...
...,...,...,...,...,...,...,...,...,...,...
478440,678,Waverley West - Chancellor,Markham Station,Weekday,2025 Oct 27 12:00:00 AM,18:30-22:30,1,0,45,678##Markham Station##20251027000000##Weekday#...
478441,678,Waverley West - Chancellor,Markham Station,Weekday,2025 Oct 27 12:00:00 AM,05:00-09:00,10,3,79,678##Markham Station##20251027000000##Weekday#...
478442,22,Portage,Polo Park,Weekday,2025 Oct 27 12:00:00 AM,09:00-16:00,52,55,108,22##Polo Park##20251027000000##Weekday##09:00-...
478443,676,North Town - Burland,Bridgwater,Weekday,2025 Oct 27 12:00:00 AM,05:00-09:00,37,0,86,676##Bridgwater##20251027000000##Weekday##05:0...


## Standardize Column Names

Column names are converted to **snake_case** and cleaned to ensure consistency and compatibility with SQL-based data modeling.


In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)
df.columns.tolist()

['route_number',
 'route_name',
 'route_destination',
 'day_type',
 'day',
 'time_period',
 'early_stops',
 'late_stops',
 'on_time_stops',
 'key']

## Convert Date Column

The original `day` column is converted into a proper datetime format and simplified to a `date` field to support daily-level analysis.


In [4]:
df["day"] = pd.to_datetime(df["day"], format="%Y %b %d %I:%M:%S %p")
df["date"] = df["day"].dt.date
df.drop(columns=["day"], inplace=True)
df

,route_number,route_name,route_destination,day_type,time_period,early_stops,late_stops,on_time_stops,key,date
0,16,Selkirk-Osborne,Tyndall Park via Manitoba,Weekday,09:00-16:00,123,266,408,16##Tyndall Park via Manitoba##20240430000000#...,2024-04-30
1,28,Brookside Express,City Hall,Weekday,16:00-18:30,24,218,187,28##City Hall##20240502000000##Weekday##16:00-...,2024-05-02
2,77,Crosstown North,Polo Park,Weekday,18:30-22:30,59,42,431,77##Polo Park##20240503000000##Weekday##18:30-...,2024-05-03
3,694,Wildwood,Seel Station via Wildwood,Weekday,09:00-16:00,4,0,12,694##Seel Station via Wildwood##20240503000000...,2024-05-03
4,676,Bridgwater/River Road,Centre Street via Bridgwater,Weekday,09:00-16:00,4,58,70,676##Centre Street via Bridgwater##20240503000...,2024-05-03
...,...,...,...,...,...,...,...,...,...,...
478440,678,Waverley West - Chancellor,Markham Station,Weekday,18:30-22:30,1,0,45,678##Markham Station##20251027000000##Weekday#...,2025-10-27
478441,678,Waverley West - Chancellor,Markham Station,Weekday,05:00-09:00,10,3,79,678##Markham Station##20251027000000##Weekday#...,2025-10-27
478442,22,Portage,Polo Park,Weekday,09:00-16:00,52,55,108,22##Polo Park##20251027000000##Weekday##09:00-...,2025-10-27
478443,676,North Town - Burland,Bridgwater,Weekday,05:00-09:00,37,0,86,676##Bridgwater##20251027000000##Weekday##05:0...,2025-10-27


In [5]:
df['date']

0         2024-04-30
1         2024-05-02
2         2024-05-03
3         2024-05-03
4         2024-05-03
             ...    
478440    2025-10-27
478441    2025-10-27
478442    2025-10-27
478443    2025-10-27
478444    2025-10-30
Name: date, Length: 478445, dtype: object

## Filter to 2024 Data

The dataset is filtered to include only records from the **2024 calendar year**, ensuring alignment with the weather dataset used later in the project.


In [6]:
df['date'] = pd.to_datetime(df['date'])
df = df[df['date'].dt.year == 2024].copy()

In [7]:
df['date'].nunique()

366

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 215322 entries, 0 to 477264
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   route_number       215322 non-null  object        
 1   route_name         215322 non-null  object        
 2   route_destination  215322 non-null  object        
 3   day_type           215322 non-null  object        
 4   time_period        215322 non-null  object        
 5   early_stops        215322 non-null  int64         
 6   late_stops         215322 non-null  object        
 7   on_time_stops      215322 non-null  object        
 8   key                215322 non-null  object        
 9   date               215322 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(8)
memory usage: 18.1+ MB


## Fix Numeric Data Types

Columns representing stop counts are converted to numeric types (`Int64`) to ensure accurate calculations and compatibility with analytical queries.


In [9]:
numeric_cols = [
    'route_number', 'early_stops',
    'late_stops', 'on_time_stops'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].astype('Int64')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 215322 entries, 0 to 477264
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   route_number       210474 non-null  Int64         
 1   route_name         215322 non-null  object        
 2   route_destination  215322 non-null  object        
 3   day_type           215322 non-null  object        
 4   time_period        215322 non-null  object        
 5   early_stops        215322 non-null  Int64         
 6   late_stops         215218 non-null  Int64         
 7   on_time_stops      214740 non-null  Int64         
 8   key                215322 non-null  object        
 9   date               215322 non-null  datetime64[ns]
dtypes: Int64(4), datetime64[ns](1), object(5)
memory usage: 18.9+ MB


## Create Total Stops Metric

A new column, `total_stops`, is created by summing early, late, and on-time stops.  
This represents the total number of observed stops for each record.


In [10]:
df['total_stops'] = df['early_stops'] + df['late_stops'] + df['on_time_stops']
df.head(3)

,route_number,route_name,route_destination,day_type,time_period,early_stops,late_stops,on_time_stops,key,date,total_stops
0,16,Selkirk-Osborne,Tyndall Park via Manitoba,Weekday,09:00-16:00,123,266,408,16##Tyndall Park via Manitoba##20240430000000#...,2024-04-30,797
1,28,Brookside Express,City Hall,Weekday,16:00-18:30,24,218,187,28##City Hall##20240502000000##Weekday##16:00-...,2024-05-02,429
2,77,Crosstown North,Polo Park,Weekday,18:30-22:30,59,42,431,77##Polo Park##20240503000000##Weekday##18:30-...,2024-05-03,532


## Create On-Time Performance Rate

The `on_time_rate` metric is calculated as:

**on-time stops ÷ total stops**

The result is rounded to two decimal places for reporting and dashboard visualization.


In [11]:
df['on_time_rate'] = round(df['on_time_stops'] / df['total_stops'], 2)
df

,route_number,route_name,route_destination,day_type,time_period,early_stops,late_stops,on_time_stops,key,date,total_stops,on_time_rate
0,16,Selkirk-Osborne,Tyndall Park via Manitoba,Weekday,09:00-16:00,123,266,408,16##Tyndall Park via Manitoba##20240430000000#...,2024-04-30,797,0.51
1,28,Brookside Express,City Hall,Weekday,16:00-18:30,24,218,187,28##City Hall##20240502000000##Weekday##16:00-...,2024-05-02,429,0.44
2,77,Crosstown North,Polo Park,Weekday,18:30-22:30,59,42,431,77##Polo Park##20240503000000##Weekday##18:30-...,2024-05-03,532,0.81
3,694,Wildwood,Seel Station via Wildwood,Weekday,09:00-16:00,4,0,12,694##Seel Station via Wildwood##20240503000000...,2024-05-03,16,0.75
4,676,Bridgwater/River Road,Centre Street via Bridgwater,Weekday,09:00-16:00,4,58,70,676##Centre Street via Bridgwater##20240503000...,2024-05-03,132,0.53
...,...,...,...,...,...,...,...,...,...,...,...,...
477225,77,Crosstown North,Kildonan Place,Weekday,22:30-05:00,6,0,0,77##Kildonan Place##20241231000000##Weekday##2...,2024-12-31,6,0.0
477229,98,Westdale - Grace Hospital,Grace Hospital,Weekday,18:30-22:30,25,0,14,98##Grace Hospital##20241231000000##Weekday##1...,2024-12-31,39,0.36
477230,38,Salter,Templeton & McPhillips,Weekday,05:00-09:00,81,9,181,38##Templeton & McPhillips##20241231000000##We...,2024-12-31,271,0.67
477231,15,Sargent-Mountain,Mountain and Fife,Weekday,18:30-22:30,0,0,15,15##Mountain and Fife##20241231000000##Weekday...,2024-12-31,15,1.0


## Remove 'key' column
The 'key' column is removed, as it is a technical identifier and not necessary.

In [12]:
df.drop(columns='key', inplace=True)

In [13]:
df = df[
    [
        "date",
        "route_number",
        "route_name",
        "route_destination",
        "day_type",
        "time_period",
        "early_stops",
        "late_stops",
        "on_time_stops",
        "total_stops",
        "on_time_rate",
    ]
]
df

,date,route_number,route_name,route_destination,day_type,time_period,early_stops,late_stops,on_time_stops,total_stops,on_time_rate
0,2024-04-30,16,Selkirk-Osborne,Tyndall Park via Manitoba,Weekday,09:00-16:00,123,266,408,797,0.51
1,2024-05-02,28,Brookside Express,City Hall,Weekday,16:00-18:30,24,218,187,429,0.44
2,2024-05-03,77,Crosstown North,Polo Park,Weekday,18:30-22:30,59,42,431,532,0.81
3,2024-05-03,694,Wildwood,Seel Station via Wildwood,Weekday,09:00-16:00,4,0,12,16,0.75
4,2024-05-03,676,Bridgwater/River Road,Centre Street via Bridgwater,Weekday,09:00-16:00,4,58,70,132,0.53
...,...,...,...,...,...,...,...,...,...,...,...
477225,2024-12-31,77,Crosstown North,Kildonan Place,Weekday,22:30-05:00,6,0,0,6,0.0
477229,2024-12-31,98,Westdale - Grace Hospital,Grace Hospital,Weekday,18:30-22:30,25,0,14,39,0.36
477230,2024-12-31,38,Salter,Templeton & McPhillips,Weekday,05:00-09:00,81,9,181,271,0.67
477231,2024-12-31,15,Sargent-Mountain,Mountain and Fife,Weekday,18:30-22:30,0,0,15,15,1.0


## Remove irregular routes
Entities with missing 'route_number' are being interpreted as exceptional events, therefore will be removed.

In [15]:
df['route_number'].isna().sum()

np.int64(4848)

In [16]:
df = df.dropna(subset=['route_number'])

In [17]:
df['route_number'].isna().sum()

np.int64(0)

## Export Clean Dataset

The cleaned dataset is exported as a CSV file to be used in the next phase of the project:  
**data modeling and SQL-based integration with weather data.**


In [16]:
df.to_csv('bus_clean.csv', index=False)